# GAM: mgcv-aligned generalized additive models

NAMpy's classical GAM backend aims to reproduce `mgcv` behavior for bases,
penalties, constraints, smoothing selection, prediction, and inference.


## Model

For an exponential-family response,

$$
g\{\mathbb E(Y\mid x)\}=\eta(x)
=\beta_0+\sum_j f_j(x_j)+\sum_r z_r\beta_r,
$$

with basis representation $f_j(x)=B_j(x)\theta_j$ and roughness penalty
$\lambda_j\theta_j^\top S_j\theta_j$. REML, ML, GCV, or UBRE selects
smoothing parameters where supported.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 180
data = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
})
data["y"] = (
    np.sin(np.pi * data["x1"])
    + 0.3 * data["x2"]
    + rng.normal(0.0, 0.12, n)
)
RUN_TRAINING = False


## Formula adapter

`GAMRegressor` and `GAMClassifier` provide sklearn-style methods while exposing
the fitted parity backend as `gam_`.


In [ ]:
from nampy.models import GAMClassifier, GAMRegressor

model = GAMRegressor(
    formula="y ~ s(x1, k=10, bs='cr') + x2",
    optimize_smoothing=True,
    smoothing_method="reml",
)
model.get_params(deep=False)

if RUN_TRAINING:
    model.fit(data)
    predictions = model.predict(data)
    r2 = model.score(data, data["y"])
    metrics = model.evaluate(data, data["y"])
    components = model.predict_components(data)
    components.validate_additive_reconstruction()
    display(model.summary())
    display(model.term_importance(data))


## GAM-specific functions

Use `standard_errors`, `lpmatrix`, `summary`, and `plot` for statistical
inference and smooth inspection. The raw `GAM` surface additionally exposes
prediction types, residuals, derivatives, and parity snapshots.


In [ ]:
if RUN_TRAINING:
    se = model.standard_errors(data)
    Xp = model.lpmatrix(data)
    terms = model.gam_.predict(data, type="terms")
    residuals = model.gam_.residuals(type="deviance")
    derivative = model.gam_.derivative(data, smooth_number=1, deriv=1)
    snapshot = model.gam_.parity_snapshot(data)
    plots = model.plot(pages=1, se=True)


## Direct backend and classification

Use `nampy.gam.GAM` when you need the raw mgcv-shaped interface. The adapters
are preferable for sklearn workflows. `GAMClassifier` supports binary targets
and adds `predict_proba` and `decision_function`.


In [ ]:
from nampy.gam import GAM

raw = GAM(
    formula="y ~ s(x1, k=10, bs='cr') + x2",
    family="gaussian",
    optimize_smoothing=True,
    smoothing_method="reml",
)

binary = (data["y"] > data["y"].median()).astype(int)
classifier = GAMClassifier(k=8, basis="cr")
if RUN_TRAINING:
    raw.fit(data=data)
    link = raw.predict(data, type="link")
    classifier.fit(data[["x1", "x2"]], binary)
    probabilities = classifier.predict_proba(data[["x1", "x2"]])


## Practical limits

Prefer formulas when term structure matters. Unsupported mgcv arguments raise
explicitly rather than degrading to approximations. Treat raw basis columns as
representation-dependent when eigenspaces are not uniquely oriented.
